# Summarization speed benchmark

Compare models on the same Walden chapters. Each configuration/repetition logs timings, tokens, prompts, and summaries to MLflow.

From the project directory:

```bash
uv sync
docker compose up -d mlflow
```

Use the project Jupyter kernel and open http://127.0.0.1:5000. If Jupyter runs in Compose, use `http://mlflow:5000` for tracking.

Run configuration and preparation before the benchmark. Model requests may incur API charges.


In [1]:
from pathlib import Path

import hashlib
import itertools
import json
import math
import os
import platform
import re
import statistics
import traceback
from collections.abc import Callable, Iterator
from time import monotonic, perf_counter, sleep
from urllib.request import Request, urlopen
from uuid import uuid4

import mlflow
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama
from langchain_openrouter import ChatOpenRouter
from tqdm.auto import tqdm
from transformers import AutoTokenizer
from utils.summarization_quality import load_model_configs

# Resolve project paths whether Jupyter starts in the root or notebook folder.
PROJECT_ROOT = next(
    (directory for directory in (Path.cwd(), *Path.cwd().parents)
     if (directory / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from the project root or one of its subfolders.")

load_dotenv(PROJECT_ROOT / ".env")


d:\Workspace\Repository\llm-summarization\.venv\Lib\site-packages\langchain_core\utils\pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


True

## Model configurations

Edit `benchmark/model-configs.json` to add models.


In [2]:
TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME", "summarization-speed-benchmark")
REPEATS = 1
BENCHMARK_GROUP = "walden-v1"
TESTING = os.getenv("SUMMARIZATION_TESTING", "false").lower() == "true"

MODEL_CONFIG_PATH = PROJECT_ROOT / "benchmark" / "model-configs.json"
MODEL_CONFIGS = load_model_configs(MODEL_CONFIG_PATH)
for config in MODEL_CONFIGS:
    if not isinstance(config.get("tokenizer"), str) or not config["tokenizer"].strip():
        raise ValueError(f"Speed benchmark requires a matching tokenizer for {config['model']} in {MODEL_CONFIG_PATH}.")
mlflow.set_tracking_uri(TRACKING_URI)

PROMPT_NAMES = {
    "system": "summarization-system-prompt",
    "chapter": "summarization-chunk-prompt",
    "final": "summarization-final-prompt",
}

PROMPTS = {
    key: mlflow.genai.load_prompt(f"prompts:/{name}@latest")
    for key, name in PROMPT_NAMES.items()
}

SYSTEM_PROMPT = PROMPTS["system"].format()
CHAPTER_PROMPT = PROMPTS["chapter"].format()
FINAL_PROMPT = PROMPTS["final"].format()

SOURCE_URL = "https://www.gutenberg.org/cache/epub/205/pg205-images.html"

if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]*", BENCHMARK_GROUP):
    raise ValueError("BENCHMARK_GROUP must use letters, digits, dots, underscores, or hyphens.")
if REPEATS < 1 or not MODEL_CONFIGS:
    raise ValueError("Set REPEATS >= 1 and provide at least one model configuration.")
for config in MODEL_CONFIGS:
    if config["provider"] not in {"ollama", "openrouter"} or not config["model"].strip():
        raise ValueError("Each configuration needs a valid provider and nonempty model name.")
    if config["max_output_tokens"] <= 0:
        raise ValueError("max_output_tokens must be positive.")
    if config["provider"] == "openrouter" and not os.getenv("OPENROUTER_API_KEY"):
        raise ValueError("OPENROUTER_API_KEY is missing from .env.")


experiment = mlflow.set_experiment(EXPERIMENT_NAME)

print(f"MLflow: {TRACKING_URI}; experiment: {EXPERIMENT_NAME}")
print(f"Benchmark group: {BENCHMARK_GROUP}")
print(f"Planned runs: {len(MODEL_CONFIGS) * REPEATS}; first five chapters only: {TESTING}")
display(pd.DataFrame(MODEL_CONFIGS))


MLflow: http://127.0.0.1:5000; experiment: summarization-speed-benchmark
Benchmark group: walden-v1
Planned runs: 3; first five chapters only: False


,base_url,num_ctx,num_gpu,reasoning,top_k,repeat_penalty,tokenizer_revision,temperature,max_output_tokens,timeout_seconds,provider,model,tokenizer
0,http://127.0.0.1:11434,65536,0,False,50,1.1,main,0,8192,1800,ollama,openbmb/minicpm5-2b,openbmb/MiniCPM5-2B
1,http://127.0.0.1:11434,65536,0,False,50,1.1,main,0,8192,1800,ollama,maternion/ling-3.0-tiny:8b-Q4_K_M,inclusionAI/Ling-3.0-tiny
2,http://127.0.0.1:11434,65536,0,False,50,1.1,main,0,8192,1800,ollama,tomng/nanbeige4.1:3b-q4_K_M,Nanbeige/Nanbeige4.1-3B


## Prepare the document


In [3]:
import itertools
from collections.abc import Callable, Iterator

from docling.document_converter import DocumentConverter
from docling_core.transforms.chunker.base import BaseChunk
from docling_core.transforms.chunker.hierarchical_chunker import HierarchicalChunker


def chunk_document(source: str, *, dropwhile: Callable[[BaseChunk], bool] = lambda c: False, takewhile: Callable[[BaseChunk], bool] = lambda c: True) -> Iterator[BaseChunk]:
    """Read the document and perform a hierarchical chunking"""
    converter = DocumentConverter()
    chunks = HierarchicalChunker().chunk(converter.convert(source=source).document)
    return itertools.takewhile(takewhile, itertools.dropwhile(dropwhile, chunks))

def merge_chunks(chunks: Iterator[BaseChunk], *, headings: Callable[[BaseChunk], list[str]] = lambda c: c.meta.headings) -> Iterator[dict[str, str]]:
    """Merge chunks having the same headings"""
    prior_headings: list[str] | None = None
    document: dict[str, str] = {}
    doc_id = 0
    for chunk in chunks:
        text = chunk.text.replace('\r\n', '\n')
        current_headings = headings(chunk)
        if prior_headings != current_headings:
            if document:
                yield document
            prior_headings = current_headings
            document = {
                'doc_id': str(doc_id := doc_id + 1),
                'title': " - ".join(current_headings),
                'text': text
            }
        else:
            document['text'] += f"\n\n{text}"
    if document:
        yield document

def chunk_dropwhile(chunk: BaseChunk) -> bool:
    """Ignore front matter prior to the book start"""
    return chunk.meta.headings != ["WALDEN", "Economy"]

def chunk_takewhile(chunk: BaseChunk) -> bool:
    """Ignore remaining chunks once we see this heading"""
    return "ON THE DUTY OF CIVIL DISOBEDIENCE" not in chunk.meta.headings

def chunk_headings(chunk: BaseChunk) -> list[str]:
    """Use the h1 and h2 (chapter) headings"""
    return chunk.meta.headings[:2]


documents = list(merge_chunks(
    chunk_document(SOURCE_URL, dropwhile=chunk_dropwhile, takewhile=chunk_takewhile),
    headings=chunk_headings,
))
if TESTING:
    documents = documents[:5]
if not documents or any(not document["text"].strip() for document in documents):
    raise ValueError("Expected nonempty Walden chapters. Check the source and heading filters.")

source_text = "\n\n".join(document["text"] for document in documents)
source_characters = len(source_text)
source_words = len(source_text.split())
dataset_sha256 = hashlib.sha256(
    json.dumps(documents, ensure_ascii=False, sort_keys=True).encode("utf-8")
).hexdigest()
prompt_sha256 = hashlib.sha256(
    json.dumps([SYSTEM_PROMPT, CHAPTER_PROMPT, FINAL_PROMPT]).encode("utf-8")
).hexdigest()
print(f"{len(documents)} chapters; {source_words:,} words; {source_characters:,} characters")
print(f"Dataset SHA256: {dataset_sha256}")


18 chapters; 106,505 words; 581,844 characters
Dataset SHA256: dddef76265c768ce5131bc9aa3dbb05beb98af463f1670e4c8b6cf16984e5feb


## Model calls and timing

Calls run sequentially without automatic retries. Records include stage, chapter, latency, token counts and their source, and finish reason. Output limits are token caps, not target summary lengths.


In [4]:
THINKING_BLOCK_PATTERN = re.compile(r"<think>.*?</think>", re.IGNORECASE | re.DOTALL)
DOCUMENT_TAG_PATTERN = re.compile(r"</?document(?:\s+[^>]*)?>", re.IGNORECASE)


def clean_summary(text):
    return DOCUMENT_TAG_PATTERN.sub("", THINKING_BLOCK_PATTERN.sub("", text)).strip()


def create_model(config):
    common = {"model": config["model"], "temperature": config["temperature"], "max_retries": 0}
    if config["provider"] == "ollama":
        # Ollama does not use the OpenRouter retry option.
        common.pop("max_retries")
        return ChatOllama(
            **common,
            base_url=config["base_url"],
            validate_model_on_init=True,
            keep_alive=-1,
            reasoning=config["reasoning"],
            num_ctx=config["num_ctx"],
            num_gpu=config["num_gpu"],
            num_predict=config["max_output_tokens"],
            top_k=config["top_k"],
            repeat_penalty=config["repeat_penalty"],
            client_kwargs={"timeout": config["timeout_seconds"]},
        )
    return ChatOpenRouter(
        **common, max_tokens=config["max_output_tokens"],
        timeout=int(config["timeout_seconds"] * 1000),
    )


def generate(model, tokenizer, config, user_prompt, call_documents):
    document_text = "\n\n".join(
        f"<document id='{doc['doc_id']}' title='{doc['title']}'>\n{doc['text']}\n</document>"
        for doc in call_documents
    )
    request_text = f"{user_prompt}\n\nSource documents:\n{document_text}"
    chat = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": request_text},
    ]
    if tokenizer.chat_template:
        estimated_input_tokens = len(tokenizer.apply_chat_template(
            chat, tokenize=True, add_generation_prompt=True,
        ))
    else:
        estimated_input_tokens = len(tokenizer.encode(
            SYSTEM_PROMPT + "\n\n" + request_text, add_special_tokens=False,
        ))
    if config["provider"] == "ollama":
        if estimated_input_tokens + config["max_output_tokens"] > config["num_ctx"]:
            raise ValueError("Input plus output budget exceeds num_ctx; use smaller chapters or a larger context.")

    started_at = perf_counter()
    response = model.invoke([
        SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=request_text),
    ])
    elapsed_seconds = perf_counter() - started_at
    content = response.content
    text = content if isinstance(content, str) else "".join(
        block if isinstance(block, str) else block.get("text", "")
        for block in content
    )
    if not clean_summary(text):
        raise ValueError("The model returned an empty final summary.")
    usage = response.usage_metadata or {}
    input_tokens = usage.get("input_tokens")
    output_tokens = usage.get("output_tokens")
    count_source = "provider" if input_tokens is not None and output_tokens is not None else "mixed_or_estimated"
    if input_tokens is None:
        input_tokens = estimated_input_tokens
    if output_tokens is None:
        output_tokens = len(tokenizer.encode(text, add_special_tokens=False))
    metadata = response.response_metadata or {}
    generation_metrics = {}
    if config["provider"] == "ollama":
        total_duration_ns = metadata.get("total_duration")
        generation_metrics.update({
            "total_duration_ns": total_duration_ns,
            "ollama_total_seconds": (
                total_duration_ns / 1e9
                if total_duration_ns is not None and total_duration_ns >= 0 else None
            ),
        })
        prompt_eval_count = metadata.get("prompt_eval_count")
        prompt_eval_cached_count = metadata.get("prompt_eval_cached_count")
        prompt_eval_duration_ns = metadata.get("prompt_eval_duration")
        prefill_tokens = prompt_eval_count
        if prompt_eval_count is not None and prompt_eval_cached_count is not None:
            prefill_tokens = (
                prompt_eval_count - prompt_eval_cached_count
                if 0 <= prompt_eval_cached_count <= prompt_eval_count else None
            )
        prefill_seconds = (
            prompt_eval_duration_ns / 1e9
            if prompt_eval_duration_ns is not None and prompt_eval_duration_ns > 0
            else None
        )
        generation_metrics.update({
            "prompt_eval_count": prompt_eval_count,
            "prompt_eval_cached_count": prompt_eval_cached_count,
            "prompt_eval_duration_ns": prompt_eval_duration_ns,
            "prefill_tokens": prefill_tokens,
            "prefill_count_source": (
                "uncached_prompt_tokens" if prompt_eval_cached_count is not None
                else "provider_prompt_eval_count"
            ),
            "prefill_seconds": prefill_seconds,
            "prefill_tokens_per_second": (
                prefill_tokens / prefill_seconds
                if prefill_tokens is not None and prefill_tokens >= 0 and prefill_seconds is not None
                else None
            ),
        })
        eval_count = metadata.get("eval_count")
        eval_duration_ns = metadata.get("eval_duration")
        generation_seconds = (
            eval_duration_ns / 1e9
            if eval_duration_ns is not None and eval_duration_ns > 0
            else None
        )
        generation_metrics.update({
            "eval_count": eval_count,
            "eval_duration_ns": eval_duration_ns,
            "generation_seconds": generation_seconds,
            "generation_tokens_per_second": (
                eval_count / generation_seconds
                if eval_count is not None and eval_count >= 0 and generation_seconds is not None
                else None
            ),
        })
    return text, {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": input_tokens + output_tokens,
        "elapsed_seconds": elapsed_seconds,
        "token_count_source": count_source,
        "finish_reason": metadata.get("finish_reason") or metadata.get("done_reason"),
        **generation_metrics,
    }


In [5]:
def run_summarization(model, tokenizer, config, call_metrics, summaries):
    # Mutable buffers preserve completed calls if a later request fails.
    with tqdm(total=len(documents) + 1, desc="Summarization", unit="call", dynamic_ncols=True) as progress:
        try:
            for index, document in enumerate(documents, start=1):
                progress.set_description(f"Chapter {index}/{len(documents)}: {document['title']}")
                output, metrics = generate(model, tokenizer, config, CHAPTER_PROMPT, [document])
                call_metrics.append({"stage": "chapter", "title": document["title"], **metrics})
                summaries.append({
                    "doc_id": document["doc_id"],
                    "title": document["title"],
                    "text": clean_summary(output),
                })
                progress.set_postfix(last_call=f"{metrics['elapsed_seconds']:.1f}s", refresh=False)
                progress.update(1)

            progress.set_description("Final synthesis")
            output, metrics = generate(model, tokenizer, config, FINAL_PROMPT, summaries)
            call_metrics.append({"stage": "final_synthesis", "title": "Walden", **metrics})
            progress.set_postfix(last_call=f"{metrics['elapsed_seconds']:.1f}s", refresh=False)
            progress.update(1)
            progress.set_description("Summarization complete")
            return clean_summary(output)
        except BaseException as error:
            stage = "Interrupted" if isinstance(error, KeyboardInterrupt) else "Failed"
            progress.set_description(f"{stage}: {progress.desc}")
            raise


def benchmark_parameters(config):
    params = {
        key: value for key, value in config.items()
        if config["provider"] == "ollama" or key not in {"num_ctx", "num_gpu"}
    }
    params.update({
        "dataset_sha256": dataset_sha256,
        "prompt_sha256": prompt_sha256,
        "chapter_count": len(documents),
        "testing": TESTING,
        "concurrency": 1,
        "warmup": "separate_prompt_before_each_run",
        "warmup_max_output_tokens": min(config.get("max_output_tokens", 64), 64),
        "client_retries": 0,
        "execution_host": platform.node(),
        "platform": platform.platform(),
    })
    if config["provider"] == "ollama":
        params.update({
            "ollama_keep_alive": -1,
            "ollama_unload": "after_each_run_confirmed_via_ps",
            "reasoning": config.get("reasoning", False),
            "top_k": config.get("top_k", 50),
            "repeat_penalty": config.get("repeat_penalty", 1.1),
            "ollama_host": config.get("base_url", os.getenv("OLLAMA_HOST", "http://127.0.0.1:11434")).rstrip("/"),
        })
    return params


def benchmark_one(model, config, config_index, repeat_index, batch_id):
    params = benchmark_parameters(config)
    # Execution order and repetition numbers must not split equivalent configurations.
    config_sha256 = hashlib.sha256(
        json.dumps(params, ensure_ascii=False, sort_keys=True).encode("utf-8")
    ).hexdigest()
    with mlflow.start_run(
        run_name=f"{config['model']}-config-{config_index}-repeat-{repeat_index}",
        experiment_id=experiment.experiment_id,
        tags={
            "batch_id": batch_id,
            "benchmark_group": BENCHMARK_GROUP,
            "run_type": "benchmark",
            "config_sha256": config_sha256,
            "timing_scope": "summarization_only",
        },
    ) as run:
        mlflow.log_params({
            **params, "config_index": config_index, "repeat_index": repeat_index,
        })
        mlflow.log_dict({
            "source_url": SOURCE_URL,
            "system": SYSTEM_PROMPT, "chapter": CHAPTER_PROMPT, "final": FINAL_PROMPT,
            "platform": platform.platform(), "python": platform.python_version(),
            "mlflow": mlflow.__version__,
            "ollama_host": params.get("ollama_host"),
        }, "config/context.json")
        tokenizer = AutoTokenizer.from_pretrained(
            config["tokenizer"], revision=config.get("tokenizer_revision", "main"),
        )
        source_tokens = len(tokenizer.encode(source_text, add_special_tokens=False))

        calls, summaries = [], []
        started_at = perf_counter()
        try:
            final_summary = run_summarization(model, tokenizer, config, calls, summaries)
        except Exception:
            elapsed = perf_counter() - started_at
            mlflow.log_metrics({"failed_calls": 1, "completed_calls": len(calls), "elapsed_before_failure_seconds": elapsed})
            mlflow.log_dict({"calls": calls}, "metrics/per_call.json")
            mlflow.log_dict({"chapters": summaries}, "outputs/partial_summaries.json")
            mlflow.log_text(traceback.format_exc(), "errors/traceback.txt")
            raise  # The run context records FAILED; the batch continues with the next run.
        elapsed = perf_counter() - started_at

        input_tokens = sum(item["input_tokens"] for item in calls)
        output_tokens = sum(item["output_tokens"] for item in calls)
        total_tokens = input_tokens + output_tokens
        latencies = sorted(item["elapsed_seconds"] for item in calls)
        metrics = {
            "source_characters": source_characters,
            "source_words": source_words,
            "source_tokens": source_tokens,
            "model_calls": len(calls),
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_model_tokens": total_tokens,
            "model_call_seconds": sum(latencies),
            "summarization_seconds": elapsed,
            "summarization_minutes": elapsed / 60,
            "total_tokens_per_second": total_tokens / elapsed,
            "output_tokens_per_second": output_tokens / elapsed,
            "source_words_per_second": source_words / elapsed,
            "pipeline_token_multiplier": total_tokens / source_tokens,
            "call_latency_median_seconds": statistics.median(latencies),
            "call_latency_p95_seconds": latencies[math.ceil(0.95 * len(latencies)) - 1],
            "truncated_calls": sum(item["finish_reason"] in {"length", "max_tokens"} for item in calls),
            "failed_calls": 0,
        }
        if config["provider"] == "ollama":
            measured_calls = sum(
                item.get("generation_tokens_per_second") is not None for item in calls
            )
            metrics["generation_measured_calls"] = measured_calls
            total_measured_calls = sum(
                item.get("ollama_total_seconds") is not None for item in calls
            )
            metrics["ollama_total_measured_calls"] = total_measured_calls
            if calls and total_measured_calls == len(calls):
                metrics["ollama_total_seconds"] = sum(item["ollama_total_seconds"] for item in calls)
                metrics["ollama_total_minutes"] = metrics["ollama_total_seconds"] / 60
            prefill_measured_calls = sum(
                item.get("prefill_tokens_per_second") is not None for item in calls
            )
            metrics["prefill_measured_calls"] = prefill_measured_calls
            if calls and prefill_measured_calls == len(calls):
                prefill_tokens = sum(item["prefill_tokens"] for item in calls)
                prefill_seconds = sum(item["prefill_seconds"] for item in calls)
                metrics.update({
                    "prefill_tokens": prefill_tokens,
                    "prefill_seconds": prefill_seconds,
                    "prefill_tokens_per_second": prefill_tokens / prefill_seconds,
                })
            # Require complete timing coverage so the rate represents the whole run.
            if calls and measured_calls == len(calls):
                generation_tokens = sum(item["eval_count"] for item in calls)
                generation_seconds = sum(item["generation_seconds"] for item in calls)
                metrics.update({
                    "generation_tokens": generation_tokens,
                    "generation_seconds": generation_seconds,
                    "generation_tokens_per_second": generation_tokens / generation_seconds,
                })
        mlflow.set_tag("token_count_source", "provider" if all(
            item["token_count_source"] == "provider" for item in calls
        ) else "mixed_or_estimated")
        tqdm.write("Saving results to MLflow...")
        # Upload after stopping the timer so tracking traffic does not inflate throughput timing.
        mlflow.log_metrics(metrics)
        mlflow.log_dict({"calls": calls}, "metrics/per_call.json")
        mlflow.log_dict({"chapters": summaries}, "outputs/chapter_summaries.json")
        mlflow.log_text(final_summary, "outputs/final_summary.txt")
        return {"run_id": run.info.run_id, "model": config["model"], **metrics}


## Warmup and cleanup helpers

This cell only defines helpers. Each run warms its model with synthetic text and up to 64 output tokens. Ollama models stay loaded during chapter calls.

After each run, cleanup unloads the current Ollama model and checks that it is gone, with a 60-second deadline. Cleanup runs on success, failure, or a normal kernel interrupt; forced shutdown cannot guarantee it. Cleanup failure stops the batch.

Setup, warmup, and cleanup are excluded from latency and token metrics. Measurements reflect a loaded model; other Ollama users can affect results.


In [6]:
def warmup_model(config):
    # Use separate text so a benchmark chapter is not prefilled by the warmup.
    warmup_config = {**config, "max_output_tokens": min(config["max_output_tokens"], 64)}
    model = create_model(warmup_config)
    response = model.invoke(
        "Summarize this short passage in two sentences: "
        "A community garden opened beside a local library. Volunteers planted vegetables "
        "and flowers, and students measured their growth each week. The harvested food "
        "was shared with nearby residents."
    )
    content = response.content
    text = content if isinstance(content, str) else "".join(
        block if isinstance(block, str) else block.get("text", "")
        for block in content
    )
    if not clean_summary(text):
        raise ValueError(f"Warmup returned an empty summary for {config['model']}.")
    print(f"Warmup response ({config['model']}): {text}", flush=True)


def ollama_request(config, path, *, payload=None, timeout_seconds=10):
    request = Request(
        config["base_url"].rstrip("/") + path,
        data=json.dumps(payload).encode("utf-8") if payload is not None else None,
        headers={"Content-Type": "application/json"},
        method="POST" if payload is not None else "GET",
    )
    with urlopen(request, timeout=timeout_seconds) as response:
        return json.load(response)


def canonical_ollama_name(name):
    return name if ":" in name.rsplit("/", 1)[-1] else name + ":latest"


def unload_ollama(config, *, timeout_seconds=60, poll_seconds=0.5):
    if config["provider"] != "ollama":
        return
    deadline = monotonic() + timeout_seconds
    print(f"Unloading Ollama model: {config['model']}", flush=True)
    response = ollama_request(
        config, "/api/generate",
        payload={"model": config["model"], "keep_alive": 0, "stream": False},
        timeout_seconds=min(10, timeout_seconds),
    )
    target = canonical_ollama_name(response.get("model") or config["model"])
    while True:
        remaining = deadline - monotonic()
        if remaining <= 0:
            raise TimeoutError(f"Ollama model {config['model']} was not confirmed unloaded within {timeout_seconds}s.")
        state = ollama_request(config, "/api/ps", timeout_seconds=min(10, remaining))
        if not isinstance(state.get("models"), list):
            raise ValueError("Ollama /api/ps did not return a models list; unload cannot be confirmed.")
        if not any(
            canonical_ollama_name(model.get("name") or model["model"]) == target
            for model in state["models"]
        ):
            print(f"Ollama model unloaded: {config['model']}", flush=True)
            return
        sleep(min(poll_seconds, max(0, deadline - monotonic())))


def benchmark_with_warmup(config, config_index, repeat_index, batch_id):
    try:
        warmup_model(config)
        model = create_model(config)
        return benchmark_one(model, config, config_index, repeat_index, batch_id)
    except Exception as error:
        return {"error": str(error)}
    finally:
        # Propagate cleanup failures to stop the batch rather than risk overlapping models.
        unload_ollama(config)


In [7]:
# from openrouter.errors import TooManyRequestsResponseError

# try:
#     response = model.invoke("Reply with OK.")
# except TooManyRequestsResponseError as error:
#     print("Error body:", error.body)
#     print("Retry-After:", error.raw_response.headers.get("retry-after"))
#     raise

## Run the benchmark

Run the helper cell first. Configurations and repetitions execute sequentially: warmup, timed summarization, then Ollama cleanup.

Each execution gets a new batch ID. `BENCHMARK_GROUP` links executions for aggregation. Warmup failures appear in `failures`; failed benchmark runs stay FAILED in MLflow and are excluded from comparisons. Interrupt the kernel once to stop and trigger cleanup.

The outer bar counts finished runs, including failures. The inner bar tracks chapter calls and final synthesis. ETA updates between requests and varies with chapter size. Uploads happen after timing stops.


In [ ]:
batch_id = uuid4().hex
results = []
failures = []
print(f"Batch: {batch_id}")
with tqdm(total=REPEATS * len(MODEL_CONFIGS), desc="Batch", unit="run", dynamic_ncols=True) as batch_progress:
    for repeat_index in range(1, REPEATS + 1):
        for config_index, config in enumerate(MODEL_CONFIGS, start=1):
            run_number = (repeat_index - 1) * len(MODEL_CONFIGS) + config_index
            batch_progress.set_postfix(model=config["model"], repeat=f"{repeat_index}/{REPEATS}")
            tqdm.write(f"Run {run_number}/{batch_progress.total}: {config['provider']}/{config['model']}, repeat {repeat_index}/{REPEATS}")
            result = benchmark_with_warmup(config, config_index, repeat_index, batch_id)
            if "error" in result:
                failures.append({"model": config["model"], "config_index": config_index, "repeat_index": repeat_index, "error": result["error"]})
                tqdm.write(f"FAILED: {result['error']}")
            else:
                results.append(result)
                if config["provider"] == "ollama":
                    generation_rate = result.get("generation_tokens_per_second")
                    speed = f"{generation_rate:.2f} tokens/s" if generation_rate is not None else "N/A"
                    prefill_rate = result.get("prefill_tokens_per_second")
                    prefill_speed = f"{prefill_rate:.2f} tokens/s" if prefill_rate is not None else "N/A"
                    server_seconds = result.get("ollama_total_seconds")
                    server_latency = f"{server_seconds:.2f}s" if server_seconds is not None else "N/A"
                    tqdm.write(f"Pipeline: {result['summarization_minutes']:.2f} min; Ollama total: {server_latency}; prefill: {prefill_speed}; generation: {speed}")
                else:
                    tqdm.write(f"{result['summarization_minutes']:.2f} min; {result['total_tokens_per_second']:.2f} total tokens/s")
            batch_progress.set_postfix(successful=len(results), failed=len(failures), refresh=False)
            batch_progress.update(1)

print(f"Finished: {len(results)} successful, {len(failures)} failed. Batch: {batch_id}")
if failures:
    display(pd.DataFrame(failures))


Batch: 928c95eb03b84070ad14d55f75c950f2


Batch:   0%|          | 0/3 [00:00<?, ?run/s]

Run 1/3: ollama/openbmb/minicpm5-2b, repeat 1/1
Warmup response (openbmb/minicpm5-2b): A community garden was established next to a local library, where volunteers grow crops together and students track plant development as part of an educational program. The produce from the garden is then distributed to surrounding neighborhoods in a sharing initiative.


config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/94.4k [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 18.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/174 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/623 [00:00<?, ?B/s]

Summarization:   0%|          | 0/19 [00:00<?, ?call/s]

## Compare successful runs across executions


In [ ]:
# def aggregate_benchmarks():
#     group_filter = f"tags.benchmark_group = '{BENCHMARK_GROUP}'"
#     runs = mlflow.search_runs(
#         experiment_ids=[experiment.experiment_id],
#         filter_string=(
#             f"{group_filter} AND tags.run_type = 'benchmark' "
#             "AND attributes.status = 'FINISHED'"
#         ),
#     )
#     if runs.empty:
#         print(f"No successful benchmark runs in group {BENCHMARK_GROUP!r}.")
#         return pd.DataFrame()

#     summaries = mlflow.search_runs(
#         experiment_ids=[experiment.experiment_id],
#         filter_string=f"{group_filter} AND tags.run_type = 'summary'",
#         order_by=["attributes.start_time DESC"],
#         output_format="list",
#     )
#     summary_ids = {}
#     for summary in summaries:
#         summary_ids.setdefault(summary.data.tags["config_sha256"], summary.info.run_id)

#     runs["metrics.summarization_minutes"] = runs["metrics.summarization_seconds"] / 60
#     runs = runs.reindex(columns=runs.columns.union(["metrics.generation_tokens_per_second", "metrics.prefill_tokens_per_second", "metrics.ollama_total_seconds"]))
#     group_columns = ["params.provider", "params.model", "tags.config_sha256"]
#     comparison = runs.groupby(group_columns).agg(
#         successful_runs=("run_id", "count"),
#         median_minutes=("metrics.summarization_minutes", "median"),
#         min_minutes=("metrics.summarization_minutes", "min"),
#         max_minutes=("metrics.summarization_minutes", "max"),
#         median_total_tokens_per_second=("metrics.total_tokens_per_second", "median"),
#         median_output_tokens_per_second=("metrics.output_tokens_per_second", "median"),
#         median_generation_tokens_per_second=("metrics.generation_tokens_per_second", "median"),
#         generation_measured_runs=("metrics.generation_tokens_per_second", "count"),
#         median_prefill_tokens_per_second=("metrics.prefill_tokens_per_second", "median"),
#         prefill_measured_runs=("metrics.prefill_tokens_per_second", "count"),
#         median_ollama_total_seconds=("metrics.ollama_total_seconds", "median"),
#         ollama_total_measured_runs=("metrics.ollama_total_seconds", "count"),
#         median_source_words_per_second=("metrics.source_words_per_second", "median"),
#         truncated_calls=("metrics.truncated_calls", "sum"),
#     ).sort_values("median_minutes")

#     summary_run_ids = {}
#     for group_key, source_runs in runs.groupby(group_columns):
#         provider, model, config_sha256 = group_key
#         metrics = {
#             key: float(value) for key, value in comparison.loc[group_key].items()
#             if pd.notna(value)
#         }
#         # Params are identical within a fingerprint, except execution order/repetition.
#         params = {
#             column.removeprefix("params."): value
#             for column, value in source_runs.iloc[0].items()
#             if column.startswith("params.")
#             and column not in {"params.config_index", "params.repeat_index"}
#             and pd.notna(value)
#         }
#         with mlflow.start_run(
#             run_id=summary_ids.get(config_sha256),
#             experiment_id=experiment.experiment_id,
#             run_name=f"Summary: {model} ({config_sha256[:8]})",
#             tags={
#                 "benchmark_group": BENCHMARK_GROUP,
#                 "run_type": "summary",
#                 "config_sha256": config_sha256,
#                 "timing_scope": "aggregate_of_summarization_only",
#             },
#         ) as summary:
#             mlflow.log_params(params)
#             mlflow.log_metrics(metrics)
#             mlflow.log_dict({
#                 "run_ids": sorted(source_runs["run_id"].tolist()),
#             }, "metrics/source_runs.json")
#             summary_run_ids[config_sha256] = summary.info.run_id

#     comparison["summary_run_id"] = [
#         summary_run_ids[config_sha256]
#         for config_sha256 in comparison.index.get_level_values("tags.config_sha256")
#     ]
#     print(f"Updated {len(comparison)} summaries for group {BENCHMARK_GROUP!r}.")
#     return comparison


# comparison = aggregate_benchmarks()
# if not comparison.empty:
#     display(comparison)
